<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта: Фактура</h2>

----

### Вариант задания 10 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс Invoice в C#, который будет представлять информацию о
фактурах за поставленные товары или оказанные услуги. На основе этого класса
разработать 2-3 производных класса, демонстрирующих принципы наследования и
полиморфизма. В каждом из классов должны быть реализованы новые атрибуты и
методы, а также переопределены некоторые методы базового класса для
демонстрации полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
public interface IInvoice
{
    string InvoiceNumber { get; }
    decimal TotalAmount { get; }
    decimal CalculateTotal();
    void PrintSummary();
}
public interface IPayable
{
    bool IsPaid { get; }
    void ProcessPayment();
}
public interface IExportable
{
    void Export();
}
public interface INotifier
{
    void SendNotification(string message);
}
public class ConsoleNotifier : INotifier
{
    public void SendNotification(string message)
    {
        Console.WriteLine($"[Уведомление (DI)] {message}");
    }
}
public class InvoicePaymentProcessor
{
    private readonly IInvoice _invoice;
    private readonly INotifier _notifier;
    public InvoicePaymentProcessor(IInvoice invoice, INotifier notifier)
    {
        _invoice = invoice ?? throw new ArgumentNullException(nameof(invoice));
        _notifier = notifier ?? throw new ArgumentNullException(nameof(notifier));
    }
    public void ExecuteProcessing()
    {
        Console.WriteLine($"\n>>> [DI Service] Начало обработки фактуры №{_invoice.InvoiceNumber}...");
        decimal total = _invoice.CalculateTotal();
        _invoice.PrintSummary();
        _notifier.SendNotification($"Фактура {_invoice.InvoiceNumber} успешно обработана на сумму {total} руб.");
    }
}
public class LineItem
{
    private string _name;
    private decimal _price;
    public string Name
    {
        get => _name;
        set => _name = string.IsNullOrWhiteSpace(value) ? "Без названия" : value;
    }
    public decimal Price
    {
        get => _price;
        set => _price = value < 0 ? 0 : value;
    }
    public LineItem()
    {
        Name = "Новая позиция";
        Price = 0;
    }
    public LineItem(string name, decimal price)
    {
        Name = name;   
        Price = price; 
    }
}
public class Invoice : IInvoice, IPayable
{
    private string _invoiceNumber;
    private DateTime _issueDate;
    private decimal _totalAmount;

    public string InvoiceNumber
    {
        get => _invoiceNumber;
        set => _invoiceNumber = string.IsNullOrWhiteSpace(value) ? "INV-000" : value;
    }

    public DateTime IssueDate
    {
        get => _issueDate;
        set => _issueDate = value;
    }

    public decimal TotalAmount
    {
        get => _totalAmount;
        protected set => _totalAmount = value < 0 ? 0 : value;
    }

    public List<LineItem> Items { get; protected set; } = new List<LineItem>(); 
    public string ClientName { get; set; } = "Частное лицо"; 
    public bool IsPaid { get; protected set; } = false;      
    public decimal DiscountPercent { get; set; } = 0;       
    public decimal TaxRate { get; set; } = 20m; // НДС (Новый атрибут)
    public Invoice()
    {
        InvoiceNumber = "INV-000";
        IssueDate = DateTime.Now;
    }
    public Invoice(string number, DateTime date) : this()
    {
        InvoiceNumber = number; 
        IssueDate = date;       
    }
    public Invoice(string number, DateTime date, string clientName) : this(number, date)
    {
        ClientName = clientName;
    }
    void IPayable.ProcessPayment()
    {
        IsPaid = true;
        Console.WriteLine($"[Явная реализация IPayable] Счёт {InvoiceNumber} проведен через банковский шлюз.");
    }

    public virtual decimal CalculateTotal()
    {
        decimal baseTotal = Items.Sum(item => item.Price);
        decimal discounted = baseTotal - (baseTotal * (DiscountPercent / 100m));
        TotalAmount = discounted + (discounted * (TaxRate / 100m));
        return TotalAmount;
    }

    public virtual void AddLine(LineItem lineItem)
    {
        if (lineItem == null) return;
        Items.Add(lineItem);
        Console.WriteLine($"[Счёт {InvoiceNumber}] Добавлена позиция: {lineItem.Name} — {lineItem.Price} руб.");
    }

    public void AddLine(string name, decimal price)
    {
        AddLine(new LineItem(name, price));
    }

    public virtual void RemoveLine(LineItem lineItem)
    {
        if (lineItem != null && Items.Remove(lineItem))
        {
            Console.WriteLine($"[Счёт {InvoiceNumber}] Удалена позиция: {lineItem.Name}");
        }
    }
    public void Pay() 
    {
        IsPaid = true;
        Console.WriteLine($"[Счёт {InvoiceNumber}] Полностью оплачен.");
    }
    public void Pay(string paymentMethod)
    {
        IsPaid = true;
        Console.WriteLine($"[Счёт {InvoiceNumber}] Оплачен способом: {paymentMethod}.");
    }
    public virtual void PrintSummary()
    {
        Console.WriteLine($"\n--- Счёт {InvoiceNumber} ---");
        Console.WriteLine($"Клиент: {ClientName} | Дата: {IssueDate.ToShortDateString()}");
        Console.WriteLine($"Статус: {(IsPaid ? "Оплачен" : "Ожидает оплаты")} | Скидка: {DiscountPercent}% | НДС: {TaxRate}%");
        Console.WriteLine($"Позиций: {Items.Count} | Итоговая сумма с НДС: {CalculateTotal()} руб.");
    }
    public void SendItemTo(LineItem item, Invoice target)
    {
        if (item == null || target == null || target == this) return;

        if (Items.Contains(item))
        {
            RemoveLine(item);
            target.AddLine(item);
            Console.WriteLine($"--> [Перенос] Позиция '{item.Name}' перенесена из '{InvoiceNumber}' в '{target.InvoiceNumber}'");
        }
    }
    public void MergeWith(Invoice other)
    {
        if (other == null || other == this) return;
        Console.WriteLine($"\n--> [Слияние] Слияние счёта '{other.InvoiceNumber}' в счёт '{this.InvoiceNumber}'...");
        List<LineItem> toMove = new List<LineItem>(other.Items);
        foreach (LineItem line in toMove)
        {
            other.SendItemTo(line, this);
        }
    }

    public void CompareTotalWith(Invoice other)
    {
        if (other == null) return;
        decimal myTotal = this.CalculateTotal();
        decimal otherTotal = other.CalculateTotal();

        Console.WriteLine($"\n--> [Сравнение] Счета '{this.InvoiceNumber}' ({myTotal} руб.) и '{other.InvoiceNumber}' ({otherTotal} руб.):");
        if (myTotal > otherTotal)
            Console.WriteLine($"    Счёт '{this.InvoiceNumber}' больше на {myTotal - otherTotal} руб.");
        else if (myTotal < otherTotal)
            Console.WriteLine($"    Счёт '{other.InvoiceNumber}' больше на {otherTotal - myTotal} руб.");
        else
            Console.WriteLine("    Суммы счетов равны.");
    }
}
public class GoodsInvoice : Invoice, IExportable
{
    public DateTime SupplyDate { get; set; }
    public string DeliveryAddress { get; set; } = "Самовывоз"; 
    public decimal ShippingCost { get; set; } = 0;            
    public double WeightKg { get; set; } = 0.0; // Новый атрибут

    public GoodsInvoice(string number, DateTime date, DateTime supplyDate, string address = "Самовывоз") 
        : base(number, date)
    {
        SupplyDate = supplyDate;
        DeliveryAddress = address;
    }
    void IExportable.Export()
    {
        Console.WriteLine($"[Явная реализация IExportable] Товарный счёт {InvoiceNumber} выгружен в формате XML для логистики. Адрес: {DeliveryAddress}.");
    }

    public override decimal CalculateTotal()
    {
        return base.CalculateTotal() + ShippingCost;
    }

    public override void AddLine(LineItem lineItem)
    {
        base.AddLine(lineItem);
        Console.WriteLine("    --> [Товар] Дата поставки: " + SupplyDate.ToShortDateString());
    }

    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Товарный счёт] Доставка: {DeliveryAddress} | Вес: {WeightKg} кг | Стоимость доставки: {ShippingCost} руб.");
    }
}
public class ServiceInvoice : Invoice
{
    public DateTime ServiceDate { get; set; }
    public string SpecialistName { get; set; } = "Не назначен"; 
    public int WarrantyDays { get; set; } = 30;                 
    public ServiceInvoice(string number, DateTime date, DateTime serviceDate, string specialist = "Не назначен") 
        : base(number, date)
    {
        ServiceDate = serviceDate;
        SpecialistName = specialist;
    }
    public override void RemoveLine(LineItem lineItem)
    {
        base.RemoveLine(lineItem);
        Console.WriteLine("    --> [Услуга] Причина удаления: Отмена услуги клиентом.");
    }
    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Счёт услуг] Специалист: {SpecialistName} | Гарантия: {WarrantyDays} дней");
    }
}
public class CombinedInvoice : Invoice
{
    public bool ReturnAllowed { get; set; }
    public string SpecialNotes { get; set; } = "Нет примечаний"; 

    public CombinedInvoice(string number, DateTime date, bool returnAllowed) : base(number, date)
    {
        ReturnAllowed = returnAllowed;
    }

    public override decimal CalculateTotal()
    {
        decimal total = base.CalculateTotal();
        if (ReturnAllowed)
        {
            total += total * 0.05m;
        }
        TotalAmount = total;
        return TotalAmount;
    }

    public override void PrintSummary()
    {
        base.PrintSummary();
        Console.WriteLine($"    [Комбинированный] Возврат разрешен: {(ReturnAllowed ? "Да" : "Нет")} | Примечания: {SpecialNotes}");
    }
}
public class InvoiceManager<T> where T : IInvoice
{
    private List<T> _invoices = new List<T>();

    public void AddInvoice(T invoice)
    {
        _invoices.Add(invoice);
        Console.WriteLine($"[Generic Менеджер] Добавлен счёт №{invoice.InvoiceNumber}");
    }

    public void ProcessAllSummaries()
    {
        Console.WriteLine("\n=== [Generic Менеджер] Печать всех зарегистрированных счетов ===");
        foreach (var inv in _invoices)
        {
            inv.PrintSummary();
        }
    }

    public decimal GetGrandTotal()
    {
        return _invoices.Sum(inv => inv.CalculateTotal());
    }
}
Console.WriteLine("=== 1. ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ ===");
GoodsInvoice goodsInv = new GoodsInvoice("Т-001", DateTime.Now, DateTime.Now.AddDays(3), "ул. Ленина, д. 10");
goodsInv.ShippingCost = 500;
goodsInv.WeightKg = 4.5;
goodsInv.ClientName = "ООО Технопром";
goodsInv.AddLine("Ноутбук", 50000);
IPayable payableInvoice = goodsInv;
payableInvoice.ProcessPayment();
IExportable exportableInvoice = goodsInv;
exportableInvoice.Export();
Console.WriteLine("\n=== 2. УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (DEPENDENCY INJECTION) ===");
INotifier notifier = new ConsoleNotifier();
InvoicePaymentProcessor processor = new InvoicePaymentProcessor(goodsInv, notifier);
processor.ExecuteProcessing();
Console.WriteLine("\n=== 3. РАБОТА С ДРУГИМИ ТИПАМИ СЧЕТОВ И GENERIC МЕНЕДЖЕРОМ ===");
ServiceInvoice serviceInv = new ServiceInvoice("У-001", DateTime.Now, DateTime.Now.AddDays(1), "Иванов И.И.");
serviceInv.DiscountPercent = 10;
serviceInv.AddLine("Настройка ПО", 2500);
CombinedInvoice combinedInv = new CombinedInvoice("К-001", DateTime.Now, true);
combinedInv.AddLine("Кабель HDMI", 1500);
InvoiceManager<IInvoice> manager = new InvoiceManager<IInvoice>();
manager.AddInvoice(goodsInv);
manager.AddInvoice(serviceInv);
manager.AddInvoice(combinedInv);
manager.ProcessAllSummaries();
Console.WriteLine($"\n Общая сумма всех счетов в менеджере: {manager.GetGrandTotal()} руб.");

=== 1. ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ ===
[Счёт Т-001] Добавлена позиция: Ноутбук — 50000 руб.
    --> [Товар] Дата поставки: 19.11.2026
[Явная реализация IPayable] Счёт Т-001 проведен через банковский шлюз.
[Явная реализация IExportable] Товарный счёт Т-001 выгружен в формате XML для логистики. Адрес: ул. Ленина, д. 10.

=== 2. УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ (DEPENDENCY INJECTION) ===

>>> [DI Service] Начало обработки фактуры №Т-001...

--- Счёт Т-001 ---
Клиент: ООО Технопром | Дата: 16.11.2026
Статус: Оплачен | Скидка: 0% | НДС: 20%
Позиций: 1 | Итоговая сумма с НДС: 60500,0 руб.
    [Товарный счёт] Доставка: ул. Ленина, д. 10 | Вес: 4,5 кг | Стоимость доставки: 500 руб.
[Уведомление (DI)] Фактура Т-001 успешно обработана на сумму 60500,0 руб.

=== 3. РАБОТА С ДРУГИМИ ТИПАМИ СЧЕТОВ И GENERIC МЕНЕДЖЕРОМ ===
[Счёт У-001] Добавлена позиция: Настройка ПО — 2500 руб.
[Счёт К-001] Добавлена позиция: Кабель HDMI — 1500 руб.
[Generic Менеджер] Добавлен счёт №Т-001
[Generic Менеджер] Добавлен счёт 